<a href="https://colab.research.google.com/github/ibnu-ahmedin/afaan-oromo-hybrid-sentiment-analysis/blob/main/Rule_Based_Model.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [ ]:
# ===========================================
#  STANDALONE RULE-BASED MODEL
# ===========================================

import pandas as pd
import re
from sklearn.metrics import accuracy_score, classification_report, confusion_matrix, ConfusionMatrixDisplay
import matplotlib.pyplot as plt

# ==========================================
# 1️⃣ LOAD DATA SPLITS
# ==========================================
train_df = pd.read_csv('/content/train.csv')
val_df   = pd.read_csv('/content/val.csv')
test_df  = pd.read_csv('/content/test.csv')

print("Train:", len(train_df))
print("Val  :", len(val_df))
print("Test :", len(test_df))

# ==========================================
# 2️⃣ PREPROCESSING
# ==========================================
def clean_text(text):
    if pd.isna(text):
        return ""
    text = str(text).lower()
    text = re.sub(r'http\S+|www\S+', '', text)
    text = re.sub(r'@\w+', '@user', text)
    text = re.sub(r'[\U0001F600-\U0001FFFF]', '', text)
    text = re.sub(r'(.)\1{2,}', r'\1\1', text)
    text = re.sub(r"[^\w\s']", '', text)
    return ' '.join(text.split())

for df in [train_df, val_df, test_df]:
    df["clean_text"] = df["Text"].apply(clean_text)

# ==========================================
# 3️⃣ LOAD LEXICON
# ==========================================
lexicon_df = pd.read_csv('/content/Lexicon_Construction.csv')
lexicon_df['Word'] = lexicon_df['Word'].str.strip().str.lower()
lexicon_df['Category'] = lexicon_df['Category'].str.strip().str.lower()

polarity_dict = {}
intensifier_dict = {}
diminisher_dict = {}
negation_set = set()
phrase_dict = {}

for _, row in lexicon_df.iterrows():
    word = row['Word']
    score = row['Score']
    cat = row['Category']

    if cat in ['positive', 'negative']:
        polarity_dict[word] = score
    elif cat == 'intensifier':
        intensifier_dict[word] = score
    elif cat == 'diminisher':
        diminisher_dict[word] = score
    elif cat == 'negation':
        negation_set.add(word)

    if ' ' in word:
        phrase_dict[word] = score

# ==========================================
# 4️⃣ TOKENIZER
# ==========================================
def tokenize(text):
    if not isinstance(text, str):
        return []
    return text.split()

# ==========================================
# 5️⃣ RULE-BASED MODEL
# ==========================================
def rule_based_sentiment(text):
    tokens = tokenize(text)
    if not tokens:
        return 1  # Neutral

    total_score = 0.0
    matched_words = set()
    text_joined = " ".join(tokens)

    # Phrase scoring
    for phrase, score in phrase_dict.items():
        if phrase in text_joined:
            total_score += score
            matched_words.update(phrase.split())

    # Word frequency
    word_counts = {}
    for w in tokens:
        word_counts[w] = word_counts.get(w, 0) + 1

    # Word-level scoring
    for i, word in enumerate(tokens):
        if word in matched_words:
            continue
        if word not in polarity_dict:
            continue

        score = polarity_dict[word]

        # Repetition control
        score *= min(1.5, word_counts[word])

        # NEGATION RULES
        if i > 0 and tokens[i-1] == "miti":
            score = -score
        elif i > 0 and tokens[i-1] in ["hin", "in"]:
            if word.endswith('u'):
                score = -score

        if i > 0 and (tokens[i-1] == "naan" or tokens[i-1].endswith('n')):
            if word.endswith(('e', 'u')):
                score = -score

        # Suffix rule
        if word.endswith("ree") and word not in polarity_dict:
            score -= 0.5

        # Intensifier / Diminisher
        if i > 0:
            prev = tokens[i-1]
            if prev in intensifier_dict:
                score *= intensifier_dict[prev]
            elif prev in diminisher_dict:
                score *= diminisher_dict[prev]

        total_score += score

    # Contrast awareness
    if any(w in {"garuu", "garu","immoo", "ammo", "ta'us", "malee"} for w in tokens):
        total_score *= 1.1

    norm_score = total_score / (len(tokens) + 1)

    # Return numeric labels (0=Neg, 1=Neu, 2=Pos)
    if abs(norm_score) < 0.22:
        return 1
    elif norm_score > 0.25:
        return 2
    elif norm_score < -0.25:
        return 0
    else:
        return 1

# ==========================================
# 6️⃣ EVALUATION ON THE SAME TEST SET
# ==========================================
y_test = test_df["label"]
preds = test_df["clean_text"].apply(rule_based_sentiment).values

print("\n" + "="*55)
print("RULE-BASED RESULTS")
print("="*55)
print("Test Size:", len(test_df))
print("Accuracy :", round(accuracy_score(y_test, preds), 4))

print("\nClassification Report:\n")
print(classification_report(y_test, preds,
                            target_names=["Negative", "Neutral", "Positive"]))

cm = confusion_matrix(y_test, preds)
ConfusionMatrixDisplay(cm, display_labels=["Negative","Neutral","Positive"]).plot(cmap="Blues")
plt.title("Rule-Based Model")
plt.show()

# Save predictions for later comparison / McNemar test
pd.DataFrame({
    "y_true": y_test.values,
    "rb_preds": preds
}).to_csv("/content/rb_preds.csv", index=False)

print("\n✅ Predictions saved as rb_preds.csv")